In [0]:
# # 📦 Data Quality Utils - Reusable Validation Functions

# ## 🎯 Purpose
# This notebook provides **reusable data quality validation functions** that can be imported by Silver and Gold notebooks.

# ---

# ## 📚 Available Functions

# ### Basic Validation Functions

# #### 1️⃣ **check_not_null(df, columns)**
# * Validates that specified columns have no NULL values
# * Returns: (status, failed_count, message)

# #### 2️⃣ **check_unique(df, columns)**
# * Validates uniqueness of primary key / composite key
# * Returns: (status, duplicate_count, message)

# #### 3️⃣ **check_referential_integrity(child_df, parent_df, child_col, parent_col)**
# * Validates foreign key relationships
# * Returns: (status, orphan_count, message)

# #### 4️⃣ **check_range(df, column, min_val, max_val)**
# * Validates values are within expected range
# * Returns: (status, out_of_range_count, message)

# #### 5️⃣ **check_expected_values(df, column, expected_values)**
# * Validates categorical values match expected set
# * Returns: (status, unexpected_count, message)

# #### 6️⃣ **print_validation_header(table_name)**
# * Pretty prints validation section header

# #### 7️⃣ **print_check_result(check_name, status, details, count)**
# * Pretty prints individual check result

# ---

# ### 🚀 Orchestrator Functions

# #### 8️⃣ **technical_validations(df, primary_key_columns, critical_columns, range_checks)**
# * **Purpose:** Execute all standard technical validations in one call
# * **Parameters:**
#   * `df`: DataFrame to validate
#   * `primary_key_columns`: List of PK columns (required)
#   * `critical_columns`: List of columns that must not be NULL (optional)
#   * `range_checks`: List of tuples `(column, min_val, max_val)` (optional)
# * **Returns:** `(validation_passed: bool, total_rows: int)`
# * **Validations:** NOT NULL (PK), UNIQUE (PK), NOT NULL (critical cols), RANGE

# #### 9️⃣ **combined_validation_result(validation_technical, validation_business)**
# * **Purpose:** Print combined result from technical and business validations
# * **Parameters:**
#   * `validation_technical`: Bool from technical validations (required)
#   * `validation_business`: Bool from business validations (optional, defaults to None)
# * **Returns:** `bool` (combined result)
# * **Note:** If `validation_business` is None, only technical validation is considered

# #### 🔟 **persist_to_delta(df, target_table_path)**
# * **Purpose:** Persist DataFrame to Delta table
# * **Parameters:**
#   * `df`: DataFrame to persist
#   * `target_table_path`: Full table path (e.g., "big_data.silver.ft_orders")
# * **Returns:** None
# * **Behavior:** Always persists with overwrite mode and overwriteSchema=true

# ---

# ## 🚀 How to Use

# ### Basic Usage (Individual Functions)

# ```python
# # In Silver/Gold notebooks:
# %run ./UTILS/utils

# # Use validation functions
# df = spark.table("big_data.silver.orders")

# status, failed, msg = check_not_null(df, ["order_id", "user_id"])
# print(f"Status: {status}, Failed: {failed}")
# ```

# ### Advanced Usage (Orchestrators) ⭐

# ```python
# # In Silver/Gold notebooks:
# %run ./UTILS/utils

# df = spark.table("big_data.bronze.orders")

# # Cell 7: Technical validations (AUTOMATED)
# validation_technical, total_rows = technical_validations(
#     df=df,
#     primary_key_columns=["order_id"],
#     critical_columns=["user_id", "order_date"],
#     range_checks=[]
# )

# # Cell 8: Business validations (MANUAL - User implements custom logic)
# print_validation_header("Business Validations")

# validation_business = True

# # Example: Pattern check
# pattern = r"^N\\d+$"
# invalid = df.filter(~F.col("order_id").rlike(pattern))
# invalid_count = invalid.count()

# if invalid_count > 0:
#     validation_business = False
#     print_check_result("PATTERN (order_id ~ N123...)", "FAIL", f"{invalid_count} rows do not match pattern", invalid_count)
# else:
#     print_check_result("PATTERN (order_id ~ N123...)", "PASS", "All values match pattern", None)

# # Example: Expected values check
# status, count, msg = check_expected_values(df, "status", ['active', 'completed', 'cancelled'])
# print_check_result("EXPECTED VALUES (status)", status, msg, count if count > 0 else None)
# if status == "FAIL":
#     validation_business = False

# print("\n" + "="*60)
# if validation_business:
#     print("SUCCESS: Business validations PASSED")
# else:
#     print("FAILURE: Business validations FAILED")
# print("="*60)

# # Cell 9: Combine results (AUTOMATED)
# validation_passed = combined_validation_result(validation_technical, validation_business)

# # Cell 10: Persistence (AUTOMATED)
# if validation_passed:
#     persist_to_delta(df, "big_data.silver.ft_orders")
# else:
#     print("\nABORTED: Validation failed - table NOT persisted")
# ```

# ---

# ## 💡 Key Design Decision

# **Why no `business_validations()` orchestrator?**

# Business validations are **highly specific** to each table and domain. A generic orchestrator would:
# * Add unnecessary abstraction
# * Make code less readable
# * Hide business logic in lambda functions

# Instead, users should:
# 1. Use `print_validation_header("Business Validations")` to start the section
# 2. Write custom validation logic using the basic functions
# 3. Track `validation_business = True/False`
# 4. Use `print_check_result()` to format output
# 5. Pass the final boolean to `combined_validation_result()`

# This approach keeps business logic **visible, explicit, and maintainable**.

# ---

# ## ⚙️ Implementation
# All functions implemented above.

In [0]:
# PySpark functions
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

In [0]:
def check_not_null(df, columns):
    """
    Check that specified columns have no NULL values.
    
    Args:
        df: Spark DataFrame
        columns: List of column names to check
    
    Returns:
        tuple: (status, failed_count, message)
    """
    total_rows = df.count()
    failed_columns = []
    
    for col in columns:
        null_count = df.filter(F.col(col).isNull()).count()
        if null_count > 0:
            failed_columns.append(f"{col} ({null_count:,} NULLs)")
    
    if failed_columns:
        return "FAIL", len(failed_columns), f"Found NULLs in: {', '.join(failed_columns)}"
    else:
        return "PASS", 0, f"All {len(columns)} columns have no NULLs"

In [0]:
def check_unique(df, columns):
    """
    Check uniqueness of primary key or composite key.
    
    Args:
        df: Spark DataFrame
        columns: List of column names that form the key
    
    Returns:
        tuple: (status, duplicate_count, message)
    """
    total_rows = df.count()
    distinct_rows = df.select(columns).distinct().count()
    duplicates = total_rows - distinct_rows
    
    key_str = ", ".join(columns)
    
    if duplicates > 0:
        return "FAIL", duplicates, f"Found {duplicates:,} duplicate rows for key ({key_str})"
    else:
        return "PASS", 0, f"Key ({key_str}) is unique"

In [0]:
def check_referential_integrity(child_df, parent_df, child_col, parent_col):
    """
    Check foreign key relationships (no orphan records).
    
    Args:
        child_df: Child DataFrame with foreign key
        parent_df: Parent DataFrame with primary key
        child_col: Foreign key column name in child
        parent_col: Primary key column name in parent
    
    Returns:
        tuple: (status, orphan_count, message)
    """
    orphans = child_df.join(
        parent_df.select(parent_col),
        child_df[child_col] == parent_df[parent_col],
        "left_anti"
    ).count()
    
    if orphans > 0:
        return "FAIL", orphans, f"Found {orphans:,} orphan records ({child_col} not in parent {parent_col})"
    else:
        return "PASS", 0, f"All {child_col} values exist in parent {parent_col}"

In [0]:
def check_range(df, column, min_val, max_val):
    """
    Check that values are within expected range.
    
    Args:
        df: Spark DataFrame
        column: Column name to check
        min_val: Minimum allowed value
        max_val: Maximum allowed value
    
    Returns:
        tuple: (status, out_of_range_count, message)
    """
    out_of_range = df.filter(
        (F.col(column) < min_val) | (F.col(column) > max_val)
    ).count()
    
    if out_of_range > 0:
        return "FAIL", out_of_range, f"{out_of_range:,} values outside range [{min_val}, {max_val}]"
    else:
        return "PASS", 0, f"All values in range [{min_val}, {max_val}]"

In [0]:
def check_expected_values(df, column, expected_values):
    """
    Check that categorical values match expected set.
    
    Args:
        df: Spark DataFrame
        column: Column name to check
        expected_values: Set or list of expected values
    
    Returns:
        tuple: (status, unexpected_count, message)
    """
    expected_set = set(expected_values)
    actual_values = df.select(column).distinct().rdd.flatMap(lambda x: x).collect()
    actual_set = set(actual_values)
    
    unexpected = actual_set - expected_set
    missing = expected_set - actual_set
    
    if unexpected or missing:
        msg_parts = []
        if unexpected:
            msg_parts.append(f"Unexpected: {unexpected}")
        if missing:
            msg_parts.append(f"Missing: {missing}")
        return "FAIL", len(unexpected), ", ".join(msg_parts)
    else:
        return "PASS", 0, f"All values match expected set: {expected_set}"

In [0]:
def print_validation_header(table_name):
    """
    Print a pretty validation section header.
    
    Args:
        table_name: Name of the table being validated
    """
    print("="*60)
    print(f"DATA QUALITY VALIDATION: {table_name}")
    print("="*60)

In [0]:
def print_check_result(check_name, status, details, count=None):
    """
    Print individual check result in a pretty format.
    
    Args:
        check_name: Name of the validation check
        status: "PASS" or "FAIL"
        details: Additional details or message
        count: Optional count (e.g., failed rows)
    """
    status_icon = "✅" if status == "PASS" else "⚠️"
    
    if count is not None:
        print(f"   {status_icon} {check_name}: {details} ({count:,} issues)")
    else:
        print(f"   {status_icon} {check_name}: {details}")

In [0]:
def technical_validations(df, primary_key_columns, critical_columns=None, range_checks=None):
    """
    Execute standard technical validations on a DataFrame.
    
    Args:
        df: Spark DataFrame to validate
        primary_key_columns: List of column names forming the primary key
        critical_columns: Optional list of columns that must not be NULL
        range_checks: Optional list of tuples (column, min_val, max_val)
    
    Returns:
        tuple: (validation_passed: bool, total_rows: int)
    """
    print_validation_header("Technical Validations")
    
    total_rows = df.count()
    print(f"\nTotal rows: {total_rows:,}")
    
    validation_passed = True
    
    # 1. NOT NULL checks (Primary Key)
    status, failed, msg = check_not_null(df, primary_key_columns)
    print_check_result("NOT NULL (PK)", status, msg, failed if failed > 0 else None)
    if status == "FAIL":
        validation_passed = False
    
    # 2. UNIQUE check (Primary Key)
    status, duplicates, msg = check_unique(df, primary_key_columns)
    print_check_result(f"UNIQUE ({', '.join(primary_key_columns)})", status, msg, duplicates if duplicates > 0 else None)
    if status == "FAIL":
        validation_passed = False
    
    # 3. NOT NULL checks (Critical Columns)
    if critical_columns:
        status, failed, msg = check_not_null(df, critical_columns)
        print_check_result("NOT NULL (Critical columns)", status, msg, failed if failed > 0 else None)
        if status == "FAIL":
            validation_passed = False
    
    # 4. RANGE checks
    if range_checks:
        for column, min_val, max_val in range_checks:
            status, out_range, msg = check_range(df, column, min_val, max_val)
            print_check_result(f"RANGE ({column}: {min_val}-{max_val})", status, msg, out_range if out_range > 0 else None)
            if status == "FAIL":
                validation_passed = False
    
    print("\n" + "="*60)
    if validation_passed:
        print("SUCCESS: Technical validations PASSED")
    else:
        print("FAILURE: Technical validations FAILED")
    print("="*60)
    
    return validation_passed, total_rows

In [0]:
def combined_validation_result(validation_technical, validation_business=None):
    """
    Print combined validation result from technical and business validations.
    
    Args:
        validation_technical: Boolean result from technical validations
        validation_business: Boolean result from business validations (optional)
    
    Returns:
        bool: Combined validation result (True if all required validations passed)
    """
    if validation_business is None:
        validation_passed = validation_technical
    else:
        validation_passed = validation_technical and validation_business

    print("\n" + "="*60)
    print("OVERALL VALIDATION RESULT")
    print("="*60)
    print(f"  Technical Validation: {'PASSED ✓' if validation_technical else 'FAILED ✗'}")
    if validation_business is not None:
        print(f"  Business Validation:  {'PASSED ✓' if validation_business else 'FAILED ✗'}")
    print("="*60)
    
    if validation_passed:
        print(f"\n✓ ALL VALIDATIONS PASSED - Ready to persist to target layer")
    else:
        print(f"\n✗ SOME VALIDATIONS FAILED - Will NOT persist")
        print("\nPlease review and fix the errors above before re-running.")
    
    print("="*60)
    
    return validation_passed

In [0]:
def persist_to_delta(df, target_table_path):
    """
    Persist DataFrame to Delta table.
    
    Args:
        df: Spark DataFrame to persist
        target_table_path: Full table path (e.g., "big_data.silver.ft_orders")
    
    Returns:
        None
    """
    print(f"Persisting to {target_table_path}...")
    
    # Write to Delta table
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table_path)
    
    # Verify persistence
    final_count = spark.table(target_table_path).count()
    
    # Success message
    print("\n" + "="*60)
    print(f"SUCCESS: Table persisted")
    print("="*60)
    print(f"\nFinal Statistics:")
    print(f"  Table: {target_table_path}")
    print(f"  Rows: {final_count:,}")
    print(f"  Format: Delta")
    print("="*60)